In [ ]:
#@title (0) setup

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, random_split
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import numpy as np
from sklearn.model_selection import train_test_split
from torch.utils.data import Subset

## (1) Build the Dataset of (A+B)%5

With A and B in the range of 0..99

In [ ]:
dataset = []

for a in range(100):
  for b in range(100):
    result = (a+b)%5
    dataset.append((a, b, result))

## (2) PyTorch MLP
**Input**: 200-dim one-hot (100 for a ⚛ 100 for b)
**Output**: 5 classes (residues 0-4)
**Hiden layers**: 32 -> ReLU -> 16 -> ReLU

Parameter buget: 7045 -  that is **70%** of the 10000 sample dataset,
so the network cannot  just memorize everything.
> Parameter count (for default 32/16):
>
>   200×32+32 = 6432
>
>   32×16+16 =  528
>
>   16× 5+ 5 =   85
>
>               ────
>
>              7045
>
> (70.4 % of 10,000 dataset)
>
> (i.e. model can't memorize this dataset)


In [ ]:
class Mod5MLP(nn.Module):
  def __init__(self, hidden1=32, hidden2=16):
    super().__init__()
    self.net = nn.Sequential(
        nn.Linear(200, hidden1),
        nn.ReLU(),
        nn.Linear(hidden1, hidden2),
        nn.ReLU(),
        nn.Linear(hidden2, 5)
    )

  def forward(self, x):
    return self.net(x)

model = Mod5MLP()
print(model)

Mod5MLP(
  (net): Sequential(
    (0): Linear(in_features=200, out_features=32, bias=True)
    (1): ReLU()
    (2): Linear(in_features=32, out_features=16, bias=True)
    (3): ReLU()
    (4): Linear(in_features=16, out_features=5, bias=True)
  )
)


## (3) Dateset Train/Validation Split (70/30)

In [ ]:
class Mod5Dataset(Dataset):
  def __init__(self, dataset):
    self.dataset = dataset

  def __len__(self):
    return len(self.dataset)

  def __getitem__(self, idx):
    a,b, result = self.dataset[idx]
    #one-hot: first 100 dim = A, next 100 dim = B
    x = torch.zeros(200)
    x[a] = 1.0
    x[100+b] = 1.0
    return x, result

In [ ]:
full_dataset = Mod5Dataset(dataset)

indices = list(range(len(full_dataset)))

train_indices, validation_indices = train_test_split(indices, test_size=0.3, random_state=42, shuffle=True)

train_dataset = Subset(full_dataset, train_indices)
validation_dataset = Subset(full_dataset, validation_indices)

print(f"Train dataset size: {len(train_dataset)}")
print(f"Validation dataset size: {len(validation_dataset)}")

Train dataset size: 7000
Validation dataset size: 3000


## (4) Training Loop

Bare-bones: AdamW, CrossEntropyLoss, batch size 32, hardcoded LearnRate.

Stops at 50 epochs or 100% validation accuracy - whichever comes first.

In [ ]:
def calc_accuracy(model, validation_loader):
  correct = 0
  total = 0

  model.eval()
  with torch.no_grad():
    for inputs, expected in validation_loader:
      outputs = model(inputs).argmax(dim=1)
      correct += (outputs == expected).sum().item()
      total += expected.size(0)
  return correct / total

In [ ]:
def train_model(model, train_ds, validation_ds, epochs=50, learning_rate=0.003, batch_size=32):
  train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True)
  validation_loader = DataLoader(validation_ds, batch_size=batch_size)

  optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)
  criterion = nn.CrossEntropyLoss()

  history = {"epoch": [], "train_loss": [], "train_acc": [], "val_acc": []}

  for epoch in range(epochs):
    model.train()
    running_loss = 0.0
    for inputs, expected in train_loader:
      optimizer.zero_grad()
      outputs = model(inputs)
      loss = criterion(outputs, expected)
      loss.backward()
      optimizer.step()
      running_loss += loss.item()

    avg_loss = running_loss / len(train_loader)
    train_acc = calc_accuracy(model, train_loader)
    val_acc = calc_accuracy(model, validation_loader)

    history["epoch"].append(epoch)
    history["train_loss"].append(avg_loss)
    history["train_acc"].append(train_acc)
    history["val_acc"].append(val_acc)

    print(f"Epoch {epoch+1}/{epochs} - Loss: {avg_loss:.4f} - Train Acc: {train_acc:.4f} - Val Acc: {val_acc:.4f}")

    if val_acc == 1.0:
      print(f"Early stopping at epoch {epoch+1} due to 100% validation accuracy.")
      break

  return history

In [ ]:
# train model
model = Mod5MLP()
history = train_model(model, train_dataset, validation_dataset)

Epoch 1/50 - Loss: 1.6103 - Train Acc: 0.2461 - Val Acc: 0.2307
Epoch 2/50 - Loss: 1.2819 - Train Acc: 0.9183 - Val Acc: 0.9027
Epoch 3/50 - Loss: 0.1567 - Train Acc: 1.0000 - Val Acc: 1.0000
Early stopping at epoch 3 due to 100% validation accuracy.


In [ ]:
# use model
def invoke_model_with_a_b(model, a, b):
  x_input = torch.zeros(200)
  x_input[a] = 1.0
  x_input[100+b] = 1.0
  model.eval()
  with torch.no_grad():
    output = model(x_input.unsqueeze(0))
    output1 = output.argmax(dim=1).item()
  return output1

print(invoke_model_with_a_b(model, 10, 25))

0


## (5) Saving and loading

In [ ]:
torch.save(model.state_dict(), '/tmp/DDN-v1.pth')

newModel = Mod5MLP()
newModel.load_state_dict(torch.load('/tmp/DDN-v1.pth'))

print(invoke_model_with_a_b(newModel, 10, 25))

0


## (6) Saving Google drive

In [ ]:
def saveToGoogleDrive(model, file_name):
  try:
    import os
    from google.colab import drive

    if not os.path.exists('/content/drive/MyDrive'):
      drive.mount('/content/drive')

    SAVE_DIR = "/content/drive/MyDrive/workshop"
    # SAVE_DIR = "workshop-v1-hf"
    os.makedirs(SAVE_DIR, exist_ok=True)

    torch.save(model.state_dict(), f'{SAVE_DIR}/{file_name}')
    print(f"Model saved to Google Drive: {SAVE_DIR}/{file_name}")
  except Exception as e:
    print(f"Error saving model to Google Drive: {e}")

saveToGoogleDrive(model, 'DDN-v1.pth')

Mounted at /content/drive
Model saved to Google Drive: /content/drive/MyDrive/workshop/DDN-v1.pth


## (7) Load Pretrained Model from HuggingFace

In [ ]:
from huggingface_hub import hf_hub_download
import torch
from google.colab import userdata

# Set device to GPU if available, else CPU
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

def load_model_from_hf(repo_id="howardyang2009/llm-workshop-v1", filename="DDN-v1.pth"):
    try:
        # Retrieve the HF_TOKEN securely from Colab's secrets manager
        hf_token = userdata.get('HF_TOKEN')
    except Exception:
        print("Warning: HF_TOKEN secret not found in Colab secrets. Attempting unauthenticated download.")
        hf_token = None

    # Download the weights file from HF using the token
    weights_path = hf_hub_download(
        repo_id=repo_id,
        filename=filename,
        token=hf_token
    )
    print(f"Downloaded weights file to: {weights_path}")

    # Initialize model architecture, map to selected device, and load state dict
    loaded_model = Mod5MLP()
    loaded_model.load_state_dict(torch.load(weights_path, map_location=device))
    loaded_model.to(device)
    loaded_model.eval()
    return loaded_model

# Load the model directly to the device (GPU/CPU)
hf_model = load_model_from_hf()

# Test with an example calculation (a=10, b=25)
a, b = 5, 15
result = invoke_model_with_a_b(hf_model, a, b)
print(f"Result of ({a} + {b}) % 5 using the HF loaded model on {device}: {result}")

Using device: cuda


DDN-v1.pth: reconstructing file:   0%|          |  0.00B / 30.9kB            

DDN-v1.pth: downloading bytes:           |  0.00B            

Downloaded weights file to: /root/.cache/huggingface/hub/models--howardyang2009--llm-workshop-v1/snapshots/564e5f94390398f3dc623bb862eca674e780dce5/DDN-v1.pth
Result of (5 + 15) % 5 using the HF loaded model on cuda: 0
